# Phase 3A: Linear Analysis - Granger Causality

## Objective

Test whether news sentiment **statistically predicts** stock returns using Granger Causality —> a linear, lag-based test that asks:

> *"Does knowing yesterday's sentiment help predict today's return, beyond what return history alone can tell us?"*

## Pipeline Overview

```
Phase 1: Data Preparation & Alignment
    ↓
Phase 2: Pre-Analysis Validation (ADF Stationarity Test)
    ↓
Phase 3: Granger Causality Loop (60 tickers × 2 models × 5 lags)
    ↓
Phase 4: Market Signal Control Test (Broad Market)
    ↓
Phase 5: Reporting & Visualization
    ↓
Output: granger_causality_results.csv
```

## Mathematical Foundation

**Granger Causality** tests whether time series X "Granger-causes" Y:

$$Y_t = \alpha + \sum_{i=1}^{p} \beta_i Y_{t-i} + \sum_{i=1}^{p} \gamma_i X_{t-i} + \epsilon_t$$

**Null Hypothesis ($H_0$)**: $\gamma_1 = \gamma_2 = ... = \gamma_p = 0$ (sentiment adds no predictive power)

**Reject $H_0$** if $p < 0.05$ → Sentiment Granger-causes returns

---

## 1. Setup: Imports and Paths

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from statsmodels.tsa.stattools import adfuller, grangercausalitytests
import statsmodels.api as sm

import yfinance as yf

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

import os
import random
from tqdm import tqdm

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', '{:.4f}'.format)

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
project_root = '/content/drive/MyDrive/market-sentiment-impact-analysis'

data_processed = os.path.join(project_root, 'data', 'processed')

print(f'Project Root: {project_root}')
print(f'Processed Data: {data_processed}')

Project Root: /content/drive/MyDrive/market-sentiment-impact-analysis
Processed Data: /content/drive/MyDrive/market-sentiment-impact-analysis/data/processed


---

## 2: Data Preparation & Alignment

### 2.1 Load Data

In [4]:
stock_returns = pd.read_csv(os.path.join(data_processed, 'stock_returns_60.csv'))
stock_returns['Date'] = pd.to_datetime(stock_returns['Date'])

print('Stock returns loaded:')
print(f'  Shape: {stock_returns.shape}')
print(f'  Date range: {stock_returns['Date'].min().date()} to {stock_returns['Date'].max().date()}')
print(f'  Unique tickers: {stock_returns['Ticker'].nunique()}')
print(f'  Columns: {list(stock_returns.columns)}')

Stock returns loaded:
  Shape: (38340, 6)
  Date range: 2018-01-03 to 2020-07-17
  Unique tickers: 60
  Columns: ['Date', 'Ticker', 'Log_Return', 'Sector', 'Beta', 'Beta_Group']


In [5]:
sentiment = pd.read_csv(os.path.join(data_processed, 'sentiment_scores_60.csv'))
sentiment['date'] = pd.to_datetime(sentiment['date'])

print(f'Sentiment scores loaded:')
print(f'  Shape: {sentiment.shape}')
print(f'  Date range: {sentiment['date'].min().date()} to {sentiment['date'].max().date()}')
print(f'  Unique tickers: {sentiment['Ticker'].nunique()}')
print(f'  Columns: {list(sentiment.columns)}')

Sentiment scores loaded:
  Shape: (44723, 7)
  Date range: 2018-01-02 to 2020-07-18
  Unique tickers: 60
  Columns: ['date', 'Ticker', 'vader_mean', 'vader_std', 'article_count', 'finbert_mean', 'finbert_std']


In [6]:
market_sentiment = pd.read_csv(os.path.join(data_processed, 'market_sentiment_general.csv'))
market_sentiment['date'] = pd.to_datetime(market_sentiment['date'])

print(f'Market sentiment loaded:')
print(f'  Shape: {market_sentiment.shape}')
print(f'  Date range: {market_sentiment['date'].min().date()} to {market_sentiment['date'].max().date()}')
print(f'  Columns: {list(market_sentiment.columns)}')

Market sentiment loaded:
  Shape: (2445, 6)
  Date range: 2018-01-02 to 2020-07-18
  Columns: ['date', 'market_vader_mean', 'market_vader_std', 'market_article_count', 'market_finbert_mean', 'market_finbert_std']


### 2.2 Weekend Handling

**Problem**: News is published 7 days a week, but stock markets only trade Monday-Friday.

Weekend news (Saturday/Sunday) has **pent-up** informational value that materializes on Monday morning's open. A naive merge would lose this signal entirely.

**Solution**: Map Saturday → Monday, Sunday → Monday, then re-aggregate.

**Monday's Aggregated Signal**:
$$S_{Monday} = \frac{N_{Sat} \cdot \bar{s}_{Sat} + N_{Sun} \cdot \bar{s}_{Sun} + N_{Mon} \cdot \bar{s}_{Mon}}{N_{Sat} + N_{Sun} + N_{Mon}}$$

Re-aggregation uses mean of means, which is appropriate when articles have equal weight.

In [7]:
def map_to_next_trading_day(date_series):
    """
    Map Saturday and Sunday dates to the following Monday.
    Monday=0, Tuesday=1, ..., Saturday=5, Sunday=6
    """
    day_of_week = date_series.dt.dayofweek

    days_to_add = pd.Series(0, index=date_series.index)
    days_to_add[day_of_week == 5] = 2
    days_to_add[day_of_week == 6] = 1

    return date_series + pd.to_timedelta(days_to_add, unit='D')

weekend_mask = sentiment['date'].dt.dayofweek >= 5
print('Weekend news in sentiment data:')
print(f'  Saturday/Sunday articles: {weekend_mask.sum():,} ({weekend_mask.sum()/len(sentiment)*100:.1f}%)')
print(f'  Weekday articles: {(~weekend_mask).sum():,}')

sentiment['date'] = map_to_next_trading_day(sentiment['date'])

weekend_mask_mkt = market_sentiment['date'].dt.dayofweek >= 5
print('\nWeekend news in market sentiment:')
print(f'  Saturday/Sunday articles: {weekend_mask_mkt.sum():,}')
market_sentiment['date'] = map_to_next_trading_day(market_sentiment['date'])

print('\n  Weekend dates mapped to next Monday')

Weekend news in sentiment data:
  Saturday/Sunday articles: 6,297 (14.1%)
  Weekday articles: 38,426

Weekend news in market sentiment:
  Saturday/Sunday articles: 254

  Weekend dates mapped to next Monday


In [8]:
# Re-aggregate sentiment after date mapping
# Saturday + Sunday + Monday news now all aggregate into Monday

sentiment = sentiment.groupby(['date', 'Ticker']).agg({
    'vader_mean': 'mean',
    'finbert_mean': 'mean',
    'article_count': 'sum',
    'vader_std': 'mean',
    'finbert_std': 'mean'
}).reset_index()

market_sentiment = market_sentiment.groupby('date').agg({
    'market_vader_mean': 'mean',
    'market_finbert_mean': 'mean',
    'market_article_count': 'sum',
    'market_vader_std': 'mean',
    'market_finbert_std': 'mean'
}).reset_index()

print('After weekend re-aggregation:')
print(f'  Sentiment rows: {len(sentiment):,}')
print(f'  Market sentiment rows: {len(market_sentiment):,}')

remaining_weekends = (sentiment['date'].dt.dayofweek >= 5).sum()
print(f'  Remaining weekend dates: {remaining_weekends}')

After weekend re-aggregation:
  Sentiment rows: 39,213
  Market sentiment rows: 2,206
  Remaining weekend dates: 0


### 2.3 Master DataFrame

In [11]:
sentiment.rename(columns={'date': 'Date'}, inplace=True)
market_sentiment.rename(columns={'date': 'Date'}, inplace=True)

sentiment['Date'] = sentiment['Date'].dt.tz_localize(None)

# Left join: stock returns (left) + ticker sentiment (right)
master_df = stock_returns.merge(
    sentiment,
    on=['Date', 'Ticker'],
    how='left'
)

print('After stock_returns + sentiment merge:')
print(f'  Shape: {master_df.shape}')
print(f'  NaN in vader_mean: {master_df['vader_mean'].isna().sum():,}')
print(f'  NaN in finbert_mean: {master_df['finbert_mean'].isna().sum():,}')

After stock_returns + sentiment merge:
  Shape: (38340, 11)
  NaN in vader_mean: 6,554
  NaN in finbert_mean: 6,554


In [12]:
# Impute NaN with 0.0 (neutral) -> No news = no new information = neutral sentiment signal
sentiment_cols = ['vader_mean', 'finbert_mean', 'article_count', 'vader_std', 'finbert_std']
master_df[sentiment_cols] = master_df[sentiment_cols].fillna(0.0)

print('After neutral imputation:')
print(f'  Remaining NaN in vader_mean: {master_df['vader_mean'].isna().sum()}')
print(f'  Remaining NaN in finbert_mean: {master_df['finbert_mean'].isna().sum()}')

no_news_pct = (master_df['article_count'] == 0).sum() / len(master_df) * 100
print(f'\n  Trading days with no news: {(master_df['article_count'] == 0).sum():,} ({no_news_pct:.1f}%)')
print(f'  Trading days with news: {(master_df['article_count'] > 0).sum():,} ({100 - no_news_pct:.1f}%)')

After neutral imputation:
  Remaining NaN in vader_mean: 0
  Remaining NaN in finbert_mean: 0

  Trading days with no news: 6,554 (17.1%)
  Trading days with news: 31,786 (82.9%)


In [14]:
master_df['Date'] = master_df['Date'].dt.tz_localize(None)
market_sentiment['Date'] = market_sentiment['Date'].dt.tz_localize(None)

master_df = master_df.merge(
    market_sentiment,
    on='Date',
    how='left'
)

market_cols = ['market_vader_mean', 'market_finbert_mean', 'market_article_count',
               'market_vader_std', 'market_finbert_std']
master_df[market_cols] = master_df[market_cols].fillna(0.0)

print('Master DataFrame:')
print(f'  Shape: {master_df.shape}')
print(f'  Columns: {list(master_df.columns)}')
print(f'  Date range: {master_df['Date'].min().date()} to {master_df['Date'].max().date()}')
print(f'  Unique tickers: {master_df['Ticker'].nunique()}')
print(f'  Remaining NaN anywhere: {master_df.isna().sum().sum()}')
print(f'\nSample rows:')
print(master_df.head())

Master DataFrame:
  Shape: (38340, 16)
  Columns: ['Date', 'Ticker', 'Log_Return', 'Sector', 'Beta', 'Beta_Group', 'vader_mean', 'finbert_mean', 'article_count', 'vader_std', 'finbert_std', 'market_vader_mean', 'market_finbert_mean', 'market_article_count', 'market_vader_std', 'market_finbert_std']
  Date range: 2018-01-03 to 2020-07-17
  Unique tickers: 60
  Remaining NaN anywhere: 0

Sample rows:
        Date Ticker  Log_Return       Sector   Beta Beta_Group  vader_mean  finbert_mean  article_count  vader_std  finbert_std  market_vader_mean  market_finbert_mean  market_article_count  market_vader_std  market_finbert_std
0 2018-01-03    AEP     -0.0085    Utilities 0.5856   Low Beta      0.0000        0.0000         0.0000     0.0000       0.0000             0.0651               0.1177               15.0000            0.3439              0.5988
1 2018-01-03    AMP     -0.0050   Financials 1.7709  High Beta      0.0000        0.0000         0.0000     0.0000       0.0000             0.

---

## 3. Pre-Analysis Validation

### 3.1 Stationarity Test (Augmented Dickey-Fuller)

Granger Causality assumes both time series are stationary. Running it on non-stationary series produces **spurious results**.

**ADF Test**:
- $H_0$: Series has a unit root (non-stationary)
- $p < 0.05$: Reject $H_0$ → Series is stationary

In [15]:
def run_adf_test(series, name):
    clean_series = series.dropna()

    result = adfuller(clean_series, autolag='AIC')
    stat = result[0]
    p_val = result[1]
    is_stationary = p_val < 0.05

    status = 'Stationary' if is_stationary else 'Non-Stationary'
    print(f'  {name}:')
    print(f'    ADF Statistic: {stat:.4f}')
    print(f'    p-value: {p_val:.6f}')
    print(f'    Result: {status}')

    return stat, p_val, is_stationary

random.seed(42)
sample_tickers = random.sample(list(master_df['Ticker'].unique()), 3)
print(f'Testing stationarity on 3 randomly selected tickers: {sample_tickers}')
print('='*70)

adf_results = {}

for ticker in sample_tickers:
    sub = master_df[master_df['Ticker'] == ticker].sort_values('Date')
    print(f'\nTicker: {ticker} ({len(sub)} trading days)')
    print('-'*40)

    stat_ret, p_ret, st_ret = run_adf_test(sub['Log_Return'], 'Log_Return')
    stat_vad, p_vad, st_vad = run_adf_test(sub['vader_mean'], 'vader_mean')
    stat_fin, p_fin, st_fin = run_adf_test(sub['finbert_mean'], 'finbert_mean')

    adf_results[ticker] = {
        'Log_Return_p': p_ret, 'Log_Return_stationary': st_ret,
        'vader_mean_p': p_vad, 'vader_mean_stationary': st_vad,
        'finbert_mean_p': p_fin, 'finbert_mean_stationary': st_fin
    }

Testing stationarity on 3 randomly selected tickers: ['OKE', 'CHD', 'AMP']

Ticker: OKE (639 trading days)
----------------------------------------
  Log_Return:
    ADF Statistic: -5.9061
    p-value: 0.000000
    Result: Stationary
  vader_mean:
    ADF Statistic: -10.1078
    p-value: 0.000000
    Result: Stationary
  finbert_mean:
    ADF Statistic: -15.4089
    p-value: 0.000000
    Result: Stationary

Ticker: CHD (639 trading days)
----------------------------------------
  Log_Return:
    ADF Statistic: -8.7989
    p-value: 0.000000
    Result: Stationary
  vader_mean:
    ADF Statistic: -25.0004
    p-value: 0.000000
    Result: Stationary
  finbert_mean:
    ADF Statistic: -24.2562
    p-value: 0.000000
    Result: Stationary

Ticker: AMP (639 trading days)
----------------------------------------
  Log_Return:
    ADF Statistic: -6.7054
    p-value: 0.000000
    Result: Stationary
  vader_mean:
    ADF Statistic: -16.1727
    p-value: 0.000000
    Result: Stationary
  finbert

In [21]:
print('\nStationarity Summary:')

all_returns_stationary = all(v['Log_Return_stationary'] for v in adf_results.values())
all_vader_stationary = all(v['vader_mean_stationary'] for v in adf_results.values())
all_finbert_stationary = all(v['finbert_mean_stationary'] for v in adf_results.values())

print(f'  Log_Return stationary across all tested: {all_returns_stationary}')
print(f'  vader_mean stationary across all tested: {all_vader_stationary}')
print(f'  finbert_mean stationary across all tested: {all_finbert_stationary}')

if not all_vader_stationary or not all_finbert_stationary:
    print('\n   Non-stationary sentiment detected — applying first difference as fallback')
    master_df = master_df.sort_values(['Ticker', 'Date'])
    master_df['vader_mean'] = master_df.groupby('Ticker')['vader_mean'].diff()
    master_df['finbert_mean'] = master_df.groupby('Ticker')['finbert_mean'].diff()
    master_df = master_df.dropna(subset=['vader_mean', 'finbert_mean'])
    print('    First difference applied — vader_mean and finbert_mean now represent ΔSENTIMENT')
else:
    print('\n  All series stationary')


Stationarity Summary:
  Log_Return stationary across all tested: True
  vader_mean stationary across all tested: True
  finbert_mean stationary across all tested: True

  All series stationary


- **Log Returns**: Stationary ($p \ll 0.05$). Log differencing removes unit root from prices by design.
- **Sentiment scores**: Stationary because bounded $[-1, +1]$ by construction. A bounded series cannot have an infinite variance.